<table>
  <tr>
    <td><div align="left"><font size="30">Ball-beam system</font></div></td>
    <td><img src="https://raw.githubusercontent.com/petercorke/bdsim/main/docs/figs/bdsim_logo.png" width="300" height="120" style="height:auto"></td>
  </tr>
</table>

**Modeling the classic ball and beam system using *bdsim***

(c) Peter Corke 2026

In [ ]:
# BDSIM_BOOTSTRAP_CELL: installs the package (Colab, JupyterLite or local), generated by sync_bootstrap.py from _bdsim_nb_bootstrap.py, do not edit

import subprocess
import sys
PACKAGE = 'bdsim'
IMPORT_NAME = 'bdsim'
GIT_URL: str | None = 'https://github.com/petercorke/bdsim'
JL_PACKAGES = []
JL_NO_DEPS: list[str] = []
JL_EXTRAS: dict[str, list[str]] = {'control': ['control']}
COLAB_PACKAGES = []

def _version_of(package: str, module) -> str:
    import importlib.metadata
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return str(getattr(module, '__version__', 'unknown'))

def _older(version: str, required: str) -> bool:
    try:
        from packaging.version import InvalidVersion, Version
        return Version(version) < Version(required)
    except (ImportError, InvalidVersion):
        return False

async def ensure_installed(extras: tuple[str, ...] | list[str]=(), packages: tuple[str, ...] | list[str]=(), min_version: str | None=None) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        for name in JL_NO_DEPS:
            await install(name, deps=False)
        for name in JL_PACKAGES:
            await install(name)
        for extra in extras:
            for name in JL_EXTRAS.get(extra, []):
                await install(name)
        for name in packages:
            await install(name)
        await install(PACKAGE)
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
        else:
            print(f'Installing {PACKAGE}...')
            requirement = PACKAGE
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement] + list(packages) + COLAB_PACKAGES, check=True)
            try:
                from matplotlib.backends import backend_registry
            except ImportError:
                pass
            else:
                clear = getattr(backend_registry, '_clear', None)
                if clear is not None:
                    clear()
            from google.colab import output
            output.enable_custom_widget_manager()
            where, colab = ('on Colab', True)
    import importlib
    module = importlib.import_module(IMPORT_NAME)
    version = _version_of(PACKAGE, module)
    if colab and min_version is not None and _older(version, min_version) and GIT_URL:
        print(f'{PACKAGE} {version} on PyPI is older than {min_version}, installing from GitHub...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'{PACKAGE} @ git+{GIT_URL}@main'], check=True)
        importlib.invalidate_caches()
        version = _version_of(PACKAGE, module)
        where += ', development version from GitHub'
    if not colab and where == 'locally':
        import importlib.metadata
        for name in packages:
            try:
                importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                print(f'This notebook needs {name}: pip install {name}')
    if min_version is not None and where in ('locally', 'on Colab') and _older(version, min_version):
        print(f'This notebook needs {PACKAGE} {min_version} or later: pip install -U {PACKAGE}')
    print(f'Running {where} using {PACKAGE} v{version}')
    return colab

COLAB = await ensure_installed(extras=['control'], min_version='1.5.0')


# Theory

A ball of mass $m$ and radius $R$ rolls along a groove in a beam.  The beam is pivoted at its centre and is tilted from the horizontal by an angle $\theta$, which is positive when the right-hand end of the beam is raised.  The position $x$ of the ball is measured along the beam from the pivot.  The beam angle $\theta$ is the system input, and the aim is to control the position of the ball.  This is a classic problem in control education because, with the beam level, the ball can rest at any position, but a small tilt makes it accelerate away and it stays away even when the beam is levelled again.

We assume that:

* the ball rolls without slipping,
* the ball experiences viscous friction along the beam, with coefficient $b$,
* the ball is a uniform solid sphere, so its moment of inertia is $J = \frac{2}{5} m R^2$,
* the radius of the ball is neglected in the geometry, that is, the centre of the ball is treated as lying on the beam, and
* the actuator is perfect: the beam angle $\theta$ is whatever we command, unaffected by the ball's mass or position.

By applying the Euler-Lagrange method, the acceleration of the ball along the beam is given by
$$
\left(m + \frac{J}{R^{2}}\right) \ddot{x} = m x \dot{\theta}^{2} - m g \sin \theta - b \dot{x} \qquad (1)
$$
where the terms on the right are, in order, the outward (centrifugal) force due to the rotation of the beam, the component of gravity along the beam, and viscous friction.  For a uniform sphere $m + J/R^2 = \frac{7}{5} m$, so
$$
\ddot{x} = \frac{5}{7} \left( x \dot{\theta}^{2} - g \sin \theta \right) - \frac{5 b}{7 m} \dot{x} \qquad (2)
$$
The factor $\frac{5}{7} \approx 0.714$ comes from the ball rolling rather than sliding.  Some of the energy gained by descending goes into spinning the ball: the translational kinetic energy is $\frac{5}{7}$ of the total and the rotational kinetic energy is the other $\frac{2}{7}$.  A rolling ball therefore accelerates down a slope at only $\frac{5}{7}$ of the rate of a block sliding without friction.

The force of the beam on the ball, normal to the beam, is
$$
N = m g \cos \theta + m x \ddot{\theta} + 2 m \dot{x} \dot{\theta} \qquad (3)
$$
the sum of the component of the weight normal to the beam and two terms due to the rotation of the beam, an Euler term proportional to $\ddot{\theta}$ and a Coriolis term.  The ball stays in contact with the beam only while $N > 0$, that is
$$
g \cos \theta + x \ddot{\theta} + 2 \dot{x} \dot{\theta} > 0 \qquad (4)
$$
A very aggressive controller can accelerate the beam downwards faster than the ball can fall, and the ball loses contact.

## Parameters

The parameter values used in this notebook are

| Quantity | Value |
| --- | --- |
| Mass of ball ($m$) | 0.1 kg
| Radius of ball ($R$) | 0.0145 m (14.5 mm)
| Moment of inertia of ball ($J = \frac{2}{5} m R^2$) | $8.4 \times 10^{-6}$ kg m²
| Half length of beam ($l$) | 0.5 m
| Viscous friction coefficient ($b$) | 0.05 N s/m
| Gravitational acceleration ($g$) | 9.81 m/s²

# Implementation with *bdsim*

We start with the standard imports, construct a simulator object, and use that to construct
a block diagram with all the `bdsim` blocks included as methods.  The ball and beam are
simulated for short times, so we set `animation_rate=10`: the scope and animation are
refreshed 10 times per second of simulated time.

In [ ]:
import math
import numpy as np
import bdsim

sim = bdsim.BDSim(animation=True, animation_rate=10)  # create simulator

bd = sim.blockdiagram()  # create an empty block diagram

Next up, we define the parameters of the system

In [ ]:
m = 0.1     # mass of ball (kg)
R = 0.0145  # radius of ball (m)
l = 0.5     # half length of beam (m)
g = 9.81    # gravity (m/s^2)
b = 0.05    # viscous friction coefficient (N s/m)

# derived parameters
J = 0.4 * m * R**2  # moment of inertia of a solid sphere (kg m^2)
M = m + J / R**2    # effective mass of the rolling ball, 7m/5 (kg)

As a quick sanity check on these numbers, with no friction a tilt of 0.1 rad should accelerate
the ball at $\frac{5}{7} g \sin 0.1$, a bit under 0.7 m/s²

In [ ]:
5 / 7 * g * math.sin(0.1)

The system can be modeled as a chain of two integrators.  The first integrates ball acceleration $\ddot{x}$ to
velocity $\dot{x}$ and the second integrates that to position $x$.  The initial position and velocity are zero.

In [ ]:
x_dot = bd.INTEGRATOR(name="x_dot")
x = bd.INTEGRATOR(name="x", inputs=[x_dot])

Note that we use the `inputs` parameter of the second integrator to connect it to the output of the first integrator.

The system starts in an equilibrium with all states zero: the beam horizontal and the ball stationary at the centre.
With the beam level, every ball position is an equilibrium.  We apply a brief tilt of 0.1 rad (5.7°) to the beam to
get the ball rolling, for $t \in [1, 1.2]$ s.

The beam angle $\theta$ is the output of a gain block with a gain of one, which gives us a single place to connect
the input.  Later we will swap the pulse for the output of a controller.

The equations of motion also need the derivatives $\dot{\theta}$ and $\ddot{\theta}$.  We compute these by feeding
$\theta$ into a cascade of two differentiators.  A pure differentiator with a transfer function of $s$ is not
implementable – it is not causal – so the `DERIV` block includes a first-order low-pass filter

$$
\frac{s}{\frac{s}{\alpha} + 1}
$$

with a pole at $s = -\alpha$.  We choose $\alpha = 10$ rad/s, a time constant of $1/\alpha = 0.1$ s.

In [ ]:
theta = bd.GAIN(1)  # beam angle (rad)
theta[0] = bd.PIECEWISE((0, 0), (1, 0.1), (1.2, 0), name="pulse")  # tilt of 0.1 rad for 0.2s

theta_dot = theta >> bd.DERIV(alpha=10, name="theta_dot")
theta_ddot = theta_dot >> bd.DERIV(alpha=10, name="theta_ddot")

We use a pair of function blocks to compute the sine and cosine of the beam angle

In [ ]:
sin_theta = theta >> bd.FUNCTION(math.sin, name="sin(theta)")
cos_theta = theta >> bd.FUNCTION(math.cos, name="cos(theta)")

Next we compute the normal force (3) of the ball on the beam.  It should be positive while the ball is supported by the beam.
If negative it means that the beam has accelerated downward from the ball, leaving it in freefall.  This would be the
result of a very aggressive controller.

In [ ]:
N = m*g*cos_theta + m*x*theta_ddot + 2*m*x_dot*theta_dot  # normal force on the ball (N)

and then the acceleration of the ball (1), which is the input to the first integrator

In [ ]:
x_dot[0] = (m*x*theta_dot**2 - m*g*sin_theta - b*x_dot) / M  # ball acceleration (m/s^2)

We create a scope block to display the ball position and the beam angle

In [ ]:
scope = bd.SCOPE(
    inputs=[x, theta],
    labels=("x (m)", r"$\theta$ (rad)"),
    loc="lower right",
)

To include a graphical animation of our ball and beam system we need to create two functions: graphics setup,
and graphics update.  We will use simple Matplotlib
patch graphics – this kind of code is tedious to write, and it can be outsourced to a coding agent.

In [ ]:
import matplotlib.patches as patches
import matplotlib.transforms as transforms


def ballbeam_init(self, fig, ax, b=l, R=R):
    # b is the half length of the beam and R the radius of the ball, bound when this
    # is defined.  The beam is 2b wide and R tall, centered on the pivot at the origin;
    # the ball sits on top of it.
    lim = 1.25 * b  # the beam stays inside this circle at any angle
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.grid(True, linestyle="--")

    # Everything is drawn in the beam frame, with the beam level.  The update
    # function rotates and slides it.
    beam = patches.Rectangle(
        (-b, -R / 2), 2 * b, R, facecolor="0.5", edgecolor="none"
    )
    ball = patches.Circle((0, R / 2 + R), R, facecolor="red", edgecolor="none")
    pivot = patches.Circle((0, 0), R / 3, facecolor="black", edgecolor="none", zorder=3)

    for patch in (beam, ball, pivot):
        ax.add_patch(patch)

    self.beam = beam
    self.ball = ball
    self.exaggeration = 1.0  # angle exaggeration, change with anim.exaggeration = 3

def ballbeam_update(self, t, u):
    x, theta = u  # ball position along the beam, beam angle (rad)
    angle = self.exaggeration * theta  # increasing angle raises the right end

    # rotate about the pivot, which is the origin
    rotate = transforms.Affine2D().rotate(angle) + self.ax.transData
    self.beam.set_transform(rotate)

    # slide the ball along the beam, then rotate it with the beam
    slide = transforms.Affine2D().translate(x, 0).rotate(angle) + self.ax.transData
    self.ball.set_transform(slide)

These two functions are passed to an `ANIMATION` block driven by the outputs of the `x` and `theta` blocks.  These inputs
are the elements of `u` passed to `ballbeam_update`.

In [ ]:
anim = bd.ANIMATION(
    init=ballbeam_init, update=ballbeam_update, nin=2, inputs=[x, theta]
)

Our system is built, we will compile it, and display a summary

In [ ]:
bd.compile()  # check the diagram
print(bd)

# bd.report_summary()

The model comprises 31 blocks, but the majority are were created implicitly (block names are prefixed by `_`) by the arithmetic expressions. A full list of all blocks in the model can be displayed by uncommenting the line above, and rerunning the cell. To create all these blocks and connect them would be both tedious and error prone.  The power of *bdsim* is that it lets
you move effortlessly between expression in terms of blocks or using Python expressions.

The information message above is because the normal force `N` is not used by any other block, we only watch it, but you could wire it to
a `SCOPE` block.

We simulate the system for 20 seconds.  The tilt of the beam accelerates the ball to the left (negative $x$) at about
0.7 m/s² for 0.2 s, leaving it moving at about 0.14 m/s.  With the beam level again nothing pushes the ball, so it coasts,
slowed only by friction.  The velocity decays exponentially with a time constant of $M/b = 2.8$ s, so the ball has travelled
about 0.38 m by the end of the run, short of the end of the beam at 0.5 m.  The normal force stays within about 1% of the
weight $m g$ of the ball, since the beam tilts so gently.

In [ ]:
out = sim.run(bd, T=20, watch=[N, x, theta])  # simulate for 20s
print(out)

# Linearization

We can linearize this non-linear system around an equilibrium operating point, here the beam level and the ball at rest at
the centre.  An operating point is a state together with the input that holds it there, so we give both to `linearize`:
all states zero, and a beam angle of zero.  The input is the output of the `pulse` block.

In [ ]:
X = bd.linearize(x=[0, 0, 0, 0], u=[0], inputs=["pulse"], full=True)
print(X)

There are four states, the ball velocity and position and the states of the two differentiators.  The differentiators
do not affect the motion of the ball in the linear model – the only place their outputs appear is in terms that are
second order in small quantities, $x \dot{\theta}^2$ – so the ball is described by the upper left $2 \times 2$ block of
$\mathbf{A}$ and the first two rows of $\mathbf{B}$.  The remaining poles are the two differentiator poles at $-\alpha$.

For small $\theta$ equation (1) becomes

$$
\ddot{x} = -\frac{m g}{M} \theta - \frac{b}{M} \dot{x}
$$

and so, for the states $(\dot{x}, x)$ and the input $\theta$

$$
\mathbf{A} = \begin{pmatrix} -b/M & 0 \\ 1 & 0 \end{pmatrix}, \qquad
\mathbf{B} = \begin{pmatrix} -m g/M \\ 0 \end{pmatrix}
$$

which we can compare with the numerical result

In [ ]:
A_hand = np.array([[-b / M, 0], [1, 0]])
B_hand = np.array([[-m * g / M], [0]])
print(np.allclose(X.A[:2, :2], A_hand), np.allclose(X.B[:2, :], B_hand))
print(X.poles)

The poles of the ball are at $s = -b/M = -0.357$, so the velocity of the ball decays with a time constant of 2.8 s, and at
$s = 0$, from the integrator that turns velocity into position.  There is nothing to restore a displaced ball to its original
position.  The input matrix $\mathbf{B}$ is negative: raising the right-hand end of the beam pushes the ball towards the left.

# Control

## Control design

We will design a linear controller using the Python Control package, which is not a dependency of `bdsim`

```
$ pip install control
```

and build an LQ state-feedback controller so that the ball position follows a square-wave setpoint.  The beam angle $\theta$ is the
control input.  The states for the design are the ball velocity and position, the part of the linearization we identified above.

In [ ]:
import control as ct

A = X.A[:2, :2]  # ball dynamics, for the states [x_dot, x]
B = X.B[:2, :]   # effect of the beam angle on the ball

To design an LQ regulator we need to define the $Q$ (penalizes state errors) and $R$ (penalizes control effort) matrices.
$Q$ must be positive semi-definite and $R$ must be positive definite -- both are typically chosen to be diagonal.
In this case we apply a high penalty to the ball position, the state we wish to control, and a small one to its velocity.
The control effort is the beam angle in radians.

In [ ]:
Q = np.diag([0.2, 10])  # weights for [x_dot, x]
R = np.array([[10]])    # weight for control input theta

To find the optimal state-feedback $\theta = -\mathbf{K}x$ we need to solve the algebraic Riccati equation

In [ ]:
K, S, E = ct.lqr(A, B, Q, R)
print(K)
print(E)

where `K` is the optimal state-feedback gain matrix, `S` is the solution to the algebraic Riccati equation, and `E`
are the eigenvalues (poles) of the closed-loop system ($A - BK$).

To solve the algebraic Riccati equation the package uses an efficient Fortran-based package `slicot` if it is available (via the Python wrapper [slycot](https://github.com/python-control/Slycot)) otherwise it falls back to a pure-Python solution implemented in SciPy.

The gain matrix $K$ has one row per input and one column per state.  Both gains are negative because $\mathbf{B}$ is negative,
and the closed-loop poles are a complex pair with a damping ratio of about 0.73.

## Converting open-loop model to closed-loop

For closed-loop simulation we can reuse most of the model that we've already built.  We'll do some surgery on the model:

* remove the existing pulse generator that we used to verify the model
* add a square-wave demand for the ball position, between -0.2 m and 0.2 m
* use the `STATE` block, which outputs the states we name in the order we name them, to feed back the ball state
* replace the scope with ones that show the setpoint

In [ ]:
bd.delete_block("pulse")  # delete the old input block, and ALL its connections
bd.delete_block(scope)    # delete the old scope block, and ALL its connections

setpoint = bd.WAVEFORM("square", min=-0.2, max=0.2, freq=0.1, unit="Hz", name="setpoint")  # ball position demand (m)

state = bd.STATE(["x_dot", "x"])  # the ball state, in the order used for the design
demand = bd.MUX(2, inputs=[bd.CONSTANT(0), setpoint])  # demanded state, at rest at the setpoint

theta[0] = -K * (state - demand)  # state feedback, the beam angle is the control input

# new scopes to watch the ball position and the beam angle
pos_scope = bd.SCOPE(inputs=[x, setpoint], labels=("x", "setpoint"), styles=("b", "k--"), loc="lower right")
angle_scope = bd.SCOPE(inputs=[theta], labels=(r"$\theta$ (rad)",))

bd.compile()  # check the diagram

The controller $-\mathbf{K}(x - x_d)$ has been written in vector form, with the `STATE` block and a `MUX` block holding
the demanded state.  The setpoint is a position, and the demanded velocity is zero.

Simulating the closed-loop system for 20 seconds shows the desired position-following behavior.  We can observe:
* a response to each change in setpoint with about 3% overshoot, that settles to within 2% in about 2.3 s,
* no steady-state error.  At rest the ball must have zero acceleration, which requires the beam to be level, and the
  control law only makes the beam level when the ball is at the setpoint,
* a large beam angle.  The control law is proportional to the position error, so a 0.4 m change in setpoint demands a
  tilt of 0.4 rad (23°) at the instant it changes,
* the ball stays in contact with the beam, the normal force falling to at worst 0.80 N against a weight of 0.98 N.

In [ ]:
out = sim.run(bd, T=20, watch=[x, theta, N, setpoint])  # simulate for 20s

## Exercises for the reader

* Plot the normal force `N`.  How large a setpoint change can the controller handle before the ball leaves the beam?
* The control input is a beam angle of up to 23°, which is more than a real beam might allow.
  * Limit the beam angle to ±10° with a `CLIP` block, and compare the response.
  * Does the clipped system remain stable for the largest setpoint changes?
* We assumed a perfect actuator:
  * Add a model of a servo motor between the controller and the beam, for example a first-order lag, and investigate how the response changes as the lag increases.
  * Add a model of a pure force actuator (a "*voice coil*" motor) and take into account the inertia of the beam and the ball's position.
* Investigate model error: change the friction coefficient $b$ or the mass of the ball in the model but not in the controller.
* Improve the control performance:
  * adjust the $Q$ and $R$ matrices
  * try a different control strategy, such as:
    * PID control,
    * model predictive control (MPC),
    * feedforward control